# 08 — Canonical Split with Image IDs

The classical notebooks saved `splits.npz` with `X_train`, `X_val`, `X_test`, but the modern CNN pipeline needs the **image IDs** corresponding to the same split.

This notebook reproduces the exact 60/20/20 stratified split from notebook 03 while carrying `image_ids` along. It also checks that the resulting `X/y` arrays match the already saved classical split.


In [1]:
import os
import sys
from pathlib import Path

# Robust project discovery: supports both
#   parent/skin_lesion/src/config.py
# and
#   project/src/config.py
_cwd = Path(os.getcwd()).resolve()
SRC_DIR = None
PROJECT_DIR = None
for _root in [_cwd, *_cwd.parents]:
    cand = _root / "skin_lesion" / "src" / "config.py"
    if cand.exists():
        SRC_DIR = cand.parent
        PROJECT_DIR = SRC_DIR.parent
        break
    cand = _root / "src" / "config.py"
    if cand.exists():
        SRC_DIR = cand.parent
        PROJECT_DIR = SRC_DIR.parent
        break

if SRC_DIR is None:
    raise FileNotFoundError(
        "Could not find src/config.py. Run this notebook from inside the project "
        "or from the parent folder containing skin_lesion/."
    )

sys.path.insert(0, str(SRC_DIR))

from config import SEED, COST_FN, COST_FP, MC_DROPOUT_T

# Use paths relative to the detected project folder. This avoids path issues
# if the zip is extracted either as skin_lesion/ or as the project root.
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIGURES_DIR = PROJECT_DIR / "results" / "figures"
TABLES_DIR = PROJECT_DIR / "results" / "tables"
MODELS_DIR = PROJECT_DIR / "results" / "models"

for d in [PROCESSED_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_DIR   : {PROJECT_DIR}")
print(f"RAW_DIR       : {RAW_DIR}")
print(f"PROCESSED_DIR : {PROCESSED_DIR}")
print(f"FIGURES_DIR   : {FIGURES_DIR}")
print(f"TABLES_DIR    : {TABLES_DIR}")
print(f"MODELS_DIR    : {MODELS_DIR}")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold


PROJECT_DIR   : /Users/thomasbove/Desktop/AIA_project/skin_lesion_project/skin_lesion
RAW_DIR       : /Users/thomasbove/Desktop/AIA_project/skin_lesion_project/skin_lesion/data/raw
PROCESSED_DIR : /Users/thomasbove/Desktop/AIA_project/skin_lesion_project/skin_lesion/data/processed
FIGURES_DIR   : /Users/thomasbove/Desktop/AIA_project/skin_lesion_project/skin_lesion/results/figures
TABLES_DIR    : /Users/thomasbove/Desktop/AIA_project/skin_lesion_project/skin_lesion/results/tables
MODELS_DIR    : /Users/thomasbove/Desktop/AIA_project/skin_lesion_project/skin_lesion/results/models


## 1 — Load HSV features and image IDs

The file `features_hsv.npz` is the canonical ordering of samples for the classical pipeline.


In [2]:
features_path = PROCESSED_DIR / "features_hsv.npz"
if not features_path.exists():
    raise FileNotFoundError(
        f"Missing {features_path}. Run 02_feature_extraction.ipynb first."
    )

data = np.load(features_path, allow_pickle=True)
X = data["X"]
y = data["y"]
image_ids = data["image_ids"].astype(str)

# Map each image to its physical lesion (lesion_id), exactly as in notebook 03,
# so the group-aware split can be reproduced identically.
subset_for_groups = pd.read_csv(PROCESSED_DIR / "balanced_subset.csv")
subset_for_groups["image_id"] = subset_for_groups["image_id"].astype(str)
id2lesion = dict(zip(subset_for_groups["image_id"], subset_for_groups["lesion_id"]))
groups = np.array([id2lesion[i] for i in image_ids])

print(f"X shape      : {X.shape}")
print(f"y shape      : {y.shape}")
print(f"image_ids    : {image_ids.shape}")
print(f"unique lesions (groups): {len(np.unique(groups))}")
print(f"Class counts : benign={(y == 0).sum()}, melanoma={(y == 1).sum()}")
print(f"First IDs    : {image_ids[:5].tolist()}")

X shape      : (2226, 96)
y shape      : (2226,)
image_ids    : (2226,)
unique lesions (groups): 1695
Class counts : benign=1113, melanoma=1113
First IDs    : ['ISIC_0025964', 'ISIC_0030623', 'ISIC_0027190', 'ISIC_0031023', 'ISIC_0028086']


## 2 — Recreate the exact stratified split

The random seed and split ratios are identical to notebook 03:

- test = 20 % of the full dataset;
- validation = 25 % of the remaining 80 %, i.e. 20 % of the full dataset;
- train = remaining 60 %.


In [3]:
# ── Group-aware, stratified split on lesion_id (identical to notebook 03) ──
# StratifiedGroupKFold keeps every lesion_id inside a single split while preserving
# the 50/50 class balance. Folds: 0 = test, 1 = val, 2-4 = train.
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = [test_idx for _, test_idx in sgkf.split(X, y, groups)]

test_idx  = folds[0]
val_idx   = folds[1]
train_idx = np.concatenate(folds[2:])

X_train, y_train, ids_train = X[train_idx], y[train_idx], image_ids[train_idx]
X_val,   y_val,   ids_val   = X[val_idx],   y[val_idx],   image_ids[val_idx]
X_test,  y_test,  ids_test  = X[test_idx],  y[test_idx],  image_ids[test_idx]

# Sanity: no lesion may appear in more than one split.
g_tr, g_va, g_te = set(groups[train_idx]), set(groups[val_idx]), set(groups[test_idx])
assert not (g_tr & g_va) and not (g_tr & g_te) and not (g_va & g_te), \
    "Lesion-id leakage detected between splits!"

for name, Xs, ys, ids in [
    ("train", X_train, y_train, ids_train),
    ("val",   X_val,   y_val,   ids_val),
    ("test",  X_test,  y_test,  ids_test),
]:
    print(
        f"{name:5s}: X={Xs.shape}, y={ys.shape}, ids={ids.shape}, "
        f"melanoma={int(ys.sum())}, benign={int((ys == 0).sum())}, "
        f"mel_frac={ys.mean():.4f}"
    )

train: X=(1335, 96), y=(1335,), ids=(1335,), melanoma=667, benign=668, mel_frac=0.4996
val  : X=(445, 96), y=(445,), ids=(445,), melanoma=223, benign=222, mel_frac=0.5011
test : X=(446, 96), y=(446,), ids=(446,), melanoma=223, benign=223, mel_frac=0.5000


## 3 — Verify compatibility with the old split

This protects the fair comparison: EfficientNet must use the same samples as the GMM baseline.


In [4]:
old_split_path = PROCESSED_DIR / "splits.npz"
if old_split_path.exists():
    old = np.load(old_split_path)
    checks = {
        "X_train": np.allclose(X_train, old["X_train"]),
        "y_train": np.array_equal(y_train, old["y_train"]),
        "X_val":   np.allclose(X_val,   old["X_val"]),
        "y_val":   np.array_equal(y_val,   old["y_val"]),
        "X_test":  np.allclose(X_test,  old["X_test"]),
        "y_test":  np.array_equal(y_test,  old["y_test"]),
    }
    print("Compatibility with splits.npz:")
    for k, v in checks.items():
        print(f"  {k:8s}: {v}")
    assert all(checks.values()), "The reconstructed split does not match splits.npz."
else:
    print("splits.npz not found. Saving a new split with IDs anyway.")


Compatibility with splits.npz:
  X_train : True
  y_train : True
  X_val   : True
  y_val   : True
  X_test  : True
  y_test  : True


## 4 — Save split IDs and a readable split table


In [5]:
out_path = PROCESSED_DIR / "splits_with_ids.npz"
np.savez_compressed(
    out_path,
    X_train=X_train, y_train=y_train, ids_train=ids_train,
    X_val=X_val,     y_val=y_val,     ids_val=ids_val,
    X_test=X_test,   y_test=y_test,   ids_test=ids_test,
)
print(f"Saved split with IDs to: {out_path}")

rows = []
for split, ids, yy in [
    ("train", ids_train, y_train),
    ("val",   ids_val,   y_val),
    ("test",  ids_test,  y_test),
]:
    for image_id, label in zip(ids, yy):
        rows.append({"image_id": image_id, "label": int(label), "split": split})

split_df = pd.DataFrame(rows)
split_csv = PROCESSED_DIR / "split_membership.csv"
split_df.to_csv(split_csv, index=False)
print(f"Saved readable split table to: {split_csv}")
print(split_df["split"].value_counts().loc[["train", "val", "test"]])


Saved split with IDs to: /Users/thomasbove/Desktop/AIA_project/skin_lesion_project/skin_lesion/data/processed/splits_with_ids.npz
Saved readable split table to: /Users/thomasbove/Desktop/AIA_project/skin_lesion_project/skin_lesion/data/processed/split_membership.csv
split
train    1335
val       445
test      446
Name: count, dtype: int64


## 5 — Optional: merge split labels into the balanced subset

This is convenient for the CNN dataset because it gives one table with `image_id`, `image_path`, `label`, and `split`.


In [6]:
subset_path = PROCESSED_DIR / "balanced_subset.csv"
if subset_path.exists():
    subset = pd.read_csv(subset_path)
    merged = subset.merge(split_df[["image_id", "split"]], on="image_id", how="inner")
    merged_path = PROCESSED_DIR / "balanced_subset_with_split.csv"
    merged.to_csv(merged_path, index=False)
    print(f"Saved: {merged_path}")
    print(merged.groupby(["split", "label"]).size().unstack(fill_value=0))
else:
    print("balanced_subset.csv not found; skipped merged table.")


Saved: /Users/thomasbove/Desktop/AIA_project/skin_lesion_project/skin_lesion/data/processed/balanced_subset_with_split.csv
label    0    1
split          
test   223  223
train  668  667
val    222  223
